## 1) 패키지 설치 & 드라이브 연결

In [1]:
!pip -q install -U "transformers>=4.43.3" "accelerate>=0.33.0" "datasets>=2.20.0" "bitsandbytes>=0.43.3" "peft>=0.12.0" "trl>=0.9.6" sentencepiece

from google.colab import drive
import os

# 1) 구글 드라이브 마운트
drive.mount('/content/drive')  # 처음 한 번 승인 필요

# 2) 저장 경로(원하는 폴더로 바꿔도 됨)
PEFT_DIR = "/content/drive/MyDrive/models/llama3-kor-blossom-8b-lora-best"
os.makedirs(PEFT_DIR, exist_ok=True)

print("Save dir:", PEFT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save dir: /content/drive/MyDrive/models/llama3-kor-blossom-8b-lora-best


## 2) 기본 설정 & 프롬프트

In [2]:
import os, json, random, re, torch
from dataclasses import dataclass

# 모델
MODEL_ID = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

# 시스템 프롬프트(영/한, 간결)
SYSTEM_PROMPT = (
    "You are an AI assistant tasked with solving a question based on a two-person conversation. "
    "Carefully read the dialogue, understand the context, and select the most appropriate answer. "
    "당신은 두 사람의 대화를 바탕으로 문제를 해결하는 AI 어시스턴트입니다. "
    "대화를 주의 깊게 읽고 문맥을 이해한 뒤, 가장 적절한 답을 선택하세요."
)

SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device=="cuda" else "")


Device: cuda NVIDIA A100-SXM4-40GB


## 3) 데이터 로드 (train/dev)

In [3]:
import requests
from datasets import Dataset, DatasetDict

train_url = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Dialogue_Inference/dataset/original_formatted/train.json"
dev_url   = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Dialogue_Inference/dataset/original_formatted/dev.json"

def load_json(url):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

train_list = load_json(train_url)
dev_list   = load_json(dev_url)

len(train_list), len(dev_list), train_list[0].keys()


(758, 151, dict_keys(['id', 'dialogue', 'category', 'question', 'answer']))

## 4) 학습/검증용 포맷팅 (채팅 템플릿)
*   User 컨텐츠: 대화/유형/문항 + “A/B/C 한 글자만 출력” 지시
*   Assistant 컨텐츠: 정답 한 글자만

In [4]:
def build_user_prompt(row):
    dialogue = row["dialogue"].strip()
    #category = row.get("category","").strip()
    question = row["question"].strip()
    user = (
        f"[Dialogue]\n{dialogue}\n\n"
        #f"[Type] {category}\n\n"
        f"[Question]\n{question}\n\n"
        "Instruction: Output only the single letter of the correct option (A/B/C)."
    )
    return user

def rows_to_chat_examples(rows):
    data=[]
    for r in rows:
        data.append({
            "messages": [
                {"role":"system","content": SYSTEM_PROMPT},
                {"role":"user",  "content": build_user_prompt(r)},
                {"role":"assistant","content": r["answer"].strip()}
            ]
        })
    return data

train_data = rows_to_chat_examples(train_list)
val_data   = rows_to_chat_examples(dev_list)

# (옵션) train 셔플
random.shuffle(train_data)

ds = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
})
ds

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 758
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 151
    })
})

## 5) 토크나이저 & 베이스 모델 로드 (일반 LoRA, 비양자화)

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# 비양자화(bfloat16)로 단일 GPU에 모두 적재합니다. (24GB에서 LoRA만 학습)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   # bf16 권장
    device_map=None               # 수동으로 cuda로 보냅니다.
).to("cuda")

base_model.config.use_cache = False  # 학습 중 비활성화 권장


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 6) 생성 기반 평가 함수 (정확도)

In [6]:
import re, torch

TERMINATORS = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

def first_choice_letter(text: str):
    m = re.search(r"[ABC]", text.strip())
    return m.group(0) if m else None

@torch.no_grad()
def evaluate_generation(model, raw_rows, max_samples=None, max_new_tokens=4, temperature=0.0):
    model.eval()
    n = len(raw_rows) if max_samples is None else min(max_samples, len(raw_rows))
    correct = 0
    total = 0
    for i in range(n):
        row = raw_rows[i]
        messages = [
            {"role":"system","content": SYSTEM_PROMPT},
            {"role":"user",  "content": build_user_prompt(row)}
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False if temperature==0.0 else True,
            temperature=temperature,
            eos_token_id=TERMINATORS,
        )
        out = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred = first_choice_letter(out)
        gold = row["answer"].strip()
        if pred == gold:
            correct += 1
        total += 1
    return (correct/total) if total>0 else 0.0


## 7) 파인튜닝 전 베이스라인 정확도 (dev)

In [7]:
baseline_acc = evaluate_generation(base_model, dev_list, max_samples=None)
print(f"[Baseline] Dev Accuracy (no fine-tuning): {baseline_acc:.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more de

[Baseline] Dev Accuracy (no fine-tuning): 0.7020


## 8) LoRA 설정 & TRL SFTTrainer (일반 LoRA)

버전별로 SFTConfig의 매개변수명이 다르다... 꼭 확인...

In [8]:
import trl, transformers, peft
print("trl:", trl.__version__) # 실행 당시 0.21.0
print("transformers:", transformers.__version__) # 실행 당시 4.55.0
print("peft:", peft.__version__) # 실행 당시 0.17.0

trl: 0.21.0
transformers: 4.55.0
peft: 0.17.0


In [9]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, TaskType

def formatting_func(example):               # ← 단일 예시(dict)
    return tokenizer.apply_chat_template(   # ← 대화 리스트 전체를 한 번에
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

peft_config = LoraConfig(
    r=64, lora_alpha=128, lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

TRAIN_OUT = "/content/llama3-kor-blossom-8b-lora"

cfg = SFTConfig(
    output_dir=TRAIN_OUT,
    num_train_epochs=100,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    save_strategy="no",
    eval_strategy="no",
    bf16=True,
    tf32=True,
    max_length=2048,
    packing=False,
    gradient_checkpointing=True,
    report_to="none",
    dataset_num_proc=2,
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    peft_config=peft_config,
    args=cfg,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)


Applying formatting function to train dataset (num_proc=2):   0%|          | 0/758 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/758 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2):   0%|          | 0/758 [00:00<?, ? examples/s]

Applying formatting function to eval dataset (num_proc=2):   0%|          | 0/151 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=2):   0%|          | 0/151 [00:00<?, ? examples/s]

Truncating eval dataset (num_proc=2):   0%|          | 0/151 [00:00<?, ? examples/s]

## 9) 콜백: 매 에포크 생성 기반 검증 → 최고 성능 저장 + 조기 종료(patience=5)

In [10]:
from transformers import TrainerCallback, TrainerControl, TrainerState
import os, torch

class GenEvalSaverCallback(TrainerCallback):
    def __init__(self, trainer, tokenizer, val_rows, patience=5, save_dir=None):
        self.trainer = trainer            # ← 생성자에서 주입
        self.tokenizer = tokenizer        # ← 생성자에서 주입
        self.val_rows = val_rows
        self.patience = patience
        self.save_dir = save_dir or os.path.join(TRAIN_OUT, "best")
        self.best_acc = -1.0
        self.no_improve_epochs = 0
        os.makedirs(self.save_dir, exist_ok=True)

    def on_epoch_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        model = kwargs["model"]  # accelerate 래핑된 PeftModel
        acc = evaluate_generation(model, self.val_rows, max_samples=None)
        print(f"[Epoch {int(state.epoch)}] Dev Accuracy: {acc:.4f}")

        # (선택) 로그
        if hasattr(self.trainer, "log"):
            self.trainer.log({"gen_dev_accuracy": acc, "epoch": state.epoch})

        if acc > self.best_acc:
            self.best_acc = acc
            self.no_improve_epochs = 0

            # ✅ 어댑터(PEFT)만 안전하게 저장
            #   콜백에서 trainer를 쓰지 않고 모델 자체 저장으로 해결
            #   (PEFT 저장 방식은 공식 포럼/이슈에서도 권장)
            #   참고: model.save_pretrained(adapter_dir) → peft_config + adapter weights 저장
            model.save_pretrained(self.save_dir)
            self.tokenizer.save_pretrained(self.save_dir)
            print(f"  -> New best! Saved PEFT adapter to: {self.save_dir}")
        else:
            self.no_improve_epochs += 1
            print(f"  -> No improvement ({self.no_improve_epochs}/{self.patience})")

        if self.no_improve_epochs >= self.patience:
            print("Early stopping triggered (no improvement).")
            control.should_training_stop = True

        return control


gen_cb = GenEvalSaverCallback(
    trainer=trainer,
    tokenizer=tokenizer,
    val_rows=dev_list,
    patience=5,
    save_dir=PEFT_DIR,
)
trainer.add_callback(gen_cb)


## 10) 학습 시작 (검증 자동 평가 & 최고 성능 저장)

In [11]:
trainer.train()
print(f"Best dev accuracy observed: {gen_cb.best_acc:.4f}")
print(f"Best adapter path: {os.path.join(TRAIN_OUT,'best')}")

Step,Training Loss
20,2.551500
40,1.772000
60,1.474300
80,1.208200
100,0.894300
120,0.521400
140,0.302900
160,0.171500
180,0.144200
200,0.121700


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more de

[Epoch 1] Dev Accuracy: 0.6954
  -> New best! Saved PEFT adapter to: /content/drive/MyDrive/models/llama3-kor-blossom-8b-lora-best


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more de

[Epoch 2] Dev Accuracy: 0.6887
  -> No improvement (1/5)


...

[Epoch 14] Dev Accuracy: 0.7881
  -> No improvement (3/5)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more de

[Epoch 15] Dev Accuracy: 0.7947
  -> No improvement (4/5)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more de

[Epoch 16] Dev Accuracy: 0.7947
  -> No improvement (5/5)
Early stopping triggered (no improvement).
Best dev accuracy observed: 0.8013
Best adapter path: /content/llama3-kor-blossom-8b-lora/best


## 11) (선택) 메모리에 남아있는 모델로 dev 성능 확인

In [12]:
final_dev_acc = evaluate_generation(trainer.model, dev_list, max_samples=None)
print(f"[Final Model in Memory] Dev Accuracy: {final_dev_acc:.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more de

[Final Model in Memory] Dev Accuracy: 0.7947


## 12) 최고 성능 가중치 로드 후 사용 예시 (일반 LoRA)

In [13]:
'''
from google.colab import drive
drive.mount('/content/drive')
'''
import os, torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 학습 시 콜백에서 저장했던 경로와 동일하게 지정하세요.
PEFT_DIR = "/content/drive/MyDrive/models/llama3-kor-blossom-8b-lora-best"
assert os.path.isdir(PEFT_DIR), f"Adapter folder not found: {PEFT_DIR}"

'''
# (권장) 저장해둔 토크나이저를 그대로 재사용 — 같은 chat template/특수토큰 유지
tokenizer = AutoTokenizer.from_pretrained(PEFT_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 베이스 모델(bf16) 로드
infer_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,                      # 예: "MLP-KTLim/llama-3-Korean-Bllossom-8B"
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
'''
infer_base = base_model # <- 주석 해제 시 삭제

# 드라이브의 LoRA 어댑터 장착
infer_model = PeftModel.from_pretrained(infer_base, PEFT_DIR)
infer_model.eval()

# 종료 토큰(모델 템플릿에 맞춰 eos와 eot를 함께 지정)
TERMINATORS = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

# (유틸) 첫 알파벳 추출 함수 — 필요 시 포함
def first_choice_letter(text: str):
    m = re.search(r"[ABC]", text.strip())
    return m.group(0) if m else None

# === 1) dev 첫 샘플로 테스트 ===
sample = dev_list[0]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": build_user_prompt(sample)},
]

# chat 템플릿으로 프롬프트 변환 (생성 모드)
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(infer_model.device)

with torch.inference_mode():
    gen = infer_model.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=False,           # 평가 시 결정론적으로
        eos_token_id=TERMINATORS,
    )

out = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("MODEL OUTPUT:", repr(out))
print("PREDICTION:", first_choice_letter(out), "| GOLD:", sample["answer"])



/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


MODEL OUTPUT: 'B'
PREDICTION: B | GOLD: B
